# Server Code

In [ ]:
import requests
import threading
import time
import json
from urllib.parse import urlencode

# Config 
ROBOT_IPS = {
    'A': '194.47.156.201',   #Robot 1 -BLUE
    'B': '194.47.156.39',   #Robot 2 - YELLOW
    'C': '194.47.156.213',  #Robot 3 -PURPLE
    'D': '194.47.156.43',   #Robot 4 - ORANGE
}

POLL_INTERVAL   = 0.5   # seconds between polling all robots
STALE_THRESHOLD = 2.0   # seconds before a report is considered stale

In [ ]:
# Shared state 

state_lock   = threading.Lock()
robot_states = {
    robot_id: {
        'red_box':  {'detected': False, 'distance': None},
        'last_seen': None
    }
    for robot_id in ROBOT_IPS
}

In [ ]:
# Commands sent TO robots 

def set_target(robot_id, global_best_id):
    """Tell a robot who the global best is so it knows which flag to follow."""
    ip = ROBOT_IPS[robot_id]
    try:
        requests.get(
            f'http://{ip}:8089/set_target?robot_id={global_best_id}',
            timeout=1
        )
    except Exception:
        print(f"[{robot_id}] Unreachable")

def stop_robots(robot_id):
    ip = ROBOT_IPS[robot_id]
    try:
        requests.get(f'http://{ip}:8089/stop', timeout=1)
    except Exception:
        print(f"[{robot_id}] Unreachable")

def start_robots(robot_id):
    ip = ROBOT_IPS[robot_id]
    try:
        requests.get(f'http://{ip}:8089/start', timeout=1)
    except Exception:
        print(f"[{robot_id}] Unreachable")

In [ ]:
# Polling 
# Each robot is polled in its own thread so one slow robot doesn't hold up the others.

def poll_robot(robot_id):
    ip = ROBOT_IPS[robot_id]
    try:
        response = requests.get(f'http://{ip}:8089/status', timeout=1)
        data     = response.json()
        with state_lock:
            robot_states[robot_id]['red_box']   = data.get('red_box', {})
            robot_states[robot_id]['last_seen'] = time.time()
    except Exception:
        pass  # robot unreachable this cycle - keep last known state

def poll_all():
    threads = [
        threading.Thread(target=poll_robot, args=(robot_id,), daemon=True)
        for robot_id in ROBOT_IPS
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

In [ ]:
# PSO fitness and global best

def get_fitness(robot_id):
    """
    Fitness = distance to red box if detected and fresh, else infinity.
    Lower is better.
    """
    with state_lock:
        state = robot_states[robot_id]
        if state['last_seen'] is None:
            return float('inf')
        if time.time() - state['last_seen'] > STALE_THRESHOLD:
            return float('inf')
        box = state['red_box']
        if box.get('detected') and box.get('distance') is not None:
            return box['distance']
        return float('inf')

def find_global_best():
    """Return the robot ID with the smallest distance to the box."""
    scores = {robot_id: get_fitness(robot_id) for robot_id in ROBOT_IPS}
    best   = min(scores, key=scores.get)
    return best if scores[best] < float('inf') else None

In [ ]:
# Intialize all robots

for robot_id in ROBOT_IPS:
    start_robots(robot_id)

In [ ]:
# Decision loop 

try:
    while True:
        # Step 1 - poll all robots in parallel
        poll_all()

        # Step 2 - find global best
        global_best = find_global_best()

        # Step 3 - log fitness table
        print("\n--- Fitness Table ---")
        for robot_id in ROBOT_IPS:
            fitness = get_fitness(robot_id)
            label   = f"{fitness:.1f}cm" if fitness < float('inf') else "no detection"
            print(f"  Robot {robot_id}: {label}")
        print(f"  Global best: {global_best}")

        # Step 4 - broadcast global best to all robots
        if global_best:
            for robot_id in ROBOT_IPS:
                threading.Thread(
                    target=set_target,
                    args=(robot_id, global_best),
                    daemon=True
                ).start()

        time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    print("Central system stopped.")


--- Fitness Table ---
  Robot A: 72.2cm
  Robot B: 179.4cm
  Robot C: 126.0cm
  Robot D: 80.0cm
  Global best: A

--- Fitness Table ---
  Robot A: 60.1cm
  Robot B: 157.9cm
  Robot C: 94.7cm
  Robot D: 69.2cm
  Global best: A

--- Fitness Table ---
  Robot A: 50.6cm
  Robot B: 134.5cm
  Robot C: 80.5cm
  Robot D: 56.6cm
  Global best: A

--- Fitness Table ---
  Robot A: 39.9cm
  Robot B: 123.3cm
  Robot C: 61.4cm
  Robot D: 48.5cm
  Global best: A

--- Fitness Table ---
  Robot A: 37.8cm
  Robot B: 86.4cm
  Robot C: 32.2cm
  Robot D: 33.8cm
  Global best: C

--- Fitness Table ---
  Robot A: 37.8cm
  Robot B: 71.3cm
  Robot C: 32.4cm
  Robot D: 33.5cm
  Global best: C

--- Fitness Table ---
  Robot A: 37.8cm
  Robot B: 54.1cm
  Robot C: 32.0cm
  Robot D: 33.5cm
  Global best: C

--- Fitness Table ---
  Robot A: 37.8cm
  Robot B: 37.1cm
  Robot C: 31.8cm
  Robot D: 33.5cm
  Global best: C

--- Fitness Table ---
  Robot A: 38.0cm
  Robot B: 33.1cm
  Robot C: 31.5cm
  Robot D: 33.5cm
  Gl

In [ ]:
#Stop command
for robot_id in ROBOT_IPS:
    stop_robots(robot_id)

[D] Unreachable
